In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [5]:
df = pd.read_csv('train.txt',sep = ';',header = None,names = ['text','emotion'])

In [6]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [7]:
df.isnull().sum()

,0
text,0
emotion,0


In [8]:
df['emotion'].unique()

array(['sadness', 'anger', 'love', 'surprise', 'fear', 'joy'],
      dtype=object)

In [9]:
unique_emotions=df['emotion'].unique()
emotion_numbers={}
i=0
for emo in unique_emotions:
  emotion_numbers[emo]=i
  i+=1
df['emotion']=df['emotion'].map(emotion_numbers)

In [10]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [11]:
df['text'] = df['text'].apply(lambda x : x.lower())

In [12]:
import string

def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))


In [13]:

df['text'] = df['text'].apply(remove_punc)

In [14]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

df['text'] = df['text'].apply(remove_numbers)

In [15]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emojis)

In [16]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [17]:
import nltk

In [18]:
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [19]:
stop_words = set(stopwords.words('english'))

In [20]:
def remove(txt):
  words = txt.split()
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)

  return ' '.join(cleaned)


In [21]:
df['text'] = df['text'].apply(remove)


In [22]:

df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.20, random_state=42)

In [24]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)


nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)


pred_bow = nb_model.predict(X_test_bow)
print(accuracy_score(y_test, pred_bow))


0.768125


In [25]:
pred_bow

array([0, 5, 0, ..., 5, 5, 0])

In [26]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)


nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)

MultinomialNB()

In [27]:
y_pred = nb2_model.predict(X_test_tfidf)

In [28]:
print(accuracy_score(y_test, y_pred))

0.6609375


In [29]:
from sklearn.linear_model import LogisticRegression

In [30]:
logistic_model = LogisticRegression(max_iter=1000)

In [31]:
logistic_model.fit(X_train_tfidf,y_train)

LogisticRegression(max_iter=1000)

In [32]:
log_pred = logistic_model.predict(X_test_tfidf)

In [33]:
print(accuracy_score(y_test,log_pred ))

0.8628125


In [46]:
user_text = input("Enter your text: ")


Enter your text: im enraged


In [47]:
user_text = user_text.lower()
user_text = remove_punc(user_text)
user_text = remove_numbers(user_text)
user_text = remove(user_text)

In [48]:
user_vector = tfidf_vectorizer.transform([user_text])

In [49]:
prediction = logistic_model.predict(user_vector)

In [50]:
reverse_mapping = {v: k for k, v in emotion_numbers.items()}
predicted_emotion = reverse_mapping[prediction[0]]
print("Predicted emotion:", predicted_emotion)

Predicted emotion: anger
